# Importing the libraries

In [94]:
import os
import requests
from bs4 import BeautifulSoup
from typing import List
from dotenv import load_dotenv
from google import genai
import gradio as gr
from google.genai.types import GenerateContentConfig

In [95]:
load_dotenv(override=True)
google_api_key = os.getenv("GEMINI_API_KEY")

In [96]:
client = genai.Client()

# Adding website scarping using Beautiful Soup

In [97]:
# A class to represent a Webpage
class Website:
    url: str
    title: str
    text: str

    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [98]:
system_assistant = "You are a assistant that analyzes a company's website contents and create a small brochure about the company for prospective customers, investors, recurits.Respond in markdown. " 

In [129]:
def stream_google(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=GenerateContentConfig(system_instruction=system_assistant),
    )
    return response.text

In [130]:
def stream_brochure(company_name, url):
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    website = Website(url)
    contents = website.get_contents()  # ⬅️ FIXED LINE
    prompt += contents
    result = stream_google(prompt)
    return result

In [131]:
view = gr.Interface(fn = stream_brochure,
                    inputs = [
                        gr.Textbox(label = "Company Name:"),
                        gr.Textbox(label = "URL: ")
                    ],
                    outputs = [gr.Markdown(label = "Brochure: ", show_copy_button = True)],
                    flagging_mode="never",
                    title = "Company Brochure Generator"
).launch()

* Running on local URL:  http://127.0.0.1:7894
* To create a public link, set `share=True` in `launch()`.
